# Multi-label emotion classification on GoEmotions
### IE624 · Generative and Agentic AI · Fine-tuning assignment

**The task.** Read a short Reddit comment and say which of 28 labels apply (27 emotions plus *neutral*). A comment can carry one emotion or several, and nobody tells us how many, so the model has to decide both *which* labels and *how many*.

**Our approach.**
1. **Fine-tune all of `roberta-large`** (an encoder-only Transformer, 355M parameters) with 28 outputs on top, one per label.
2. **Treat each label as its own yes/no question.** Each output goes through a *sigmoid* (not a softmax), so labels don't compete and several can be "on" at the same time. The loss is binary cross-entropy, the standard objective for multi-label problems.
3. **Learn a decision rule on the validation set:** one tuned threshold per label, plus a fallback for comments where no label clears its threshold.

**Designed for a limited GPU budget.** The whole notebook runs in roughly 20–25 minutes on a single Kaggle T4. Batches group comments of similar length (less wasted padding), the best snapshot is kept as training goes, and a run that collapses early is restarted automatically.

**Ground rules we follow** (so a reader can check us):

| Rule | How this notebook respects it |
|---|---|
| Encoder-only model, encoder must be adapted | `roberta-large`; every weight is trained (full fine-tuning) |
| One GPU, at most 16 GB | One T4 (15.6 GB); the second Kaggle GPU is hidden before PyTorch starts; peak memory is logged |
| Train only on the official training split | Training uses `train` only. `validation` is used only to choose the checkpoint and the decision rule |
| Never use test labels during development | The test split is not even loaded until the optional, clearly marked last cell (Part 8) |
| One final model, no ensembles | Exactly one set of weights is shipped |

**Map of the notebook**

| Part | What happens | GPU? |
|---|---|---|
| 0 | Setup, library versions, configuration | yes |
| 1 | Data: loading, a look at the labels, tokenisation, batching | no |
| 2 | Model, parameter counts, and why we fine-tune the whole encoder | no |
| 3 | Metrics and decision-rule toolkit | no |
| 4 | Training | yes |
| 5 | The decision rule (thresholds and fallback), tuned on validation | no |
| 6 | Saving and zipping the final checkpoint | no |
| 7 | **Inference section**: self-contained, runs in a fresh session | optional |
| 8 | One-time test evaluation (off by default) | optional |

**How to run on Kaggle:** set Accelerator = *GPU T4 x2* (we use only one of the two GPUs) and Internet = *On*, then use **Save Version → Save & Run All** so training keeps going in the background. The final checkpoint ends up in `/kaggle/working/goemotions_final.zip`.

## Part 0 · Setup

Kaggle's "T4 x2" machine shows PyTorch two GPUs. The assignment allows one, so the first thing we do, *before* PyTorch is imported, is make only GPU 0 visible. The asserts below would stop the notebook if that ever failed.

We also print every library version, so the run can be reproduced later.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"        # expose exactly one GPU (must happen before torch is imported)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc, hashlib, json, math, platform, random, shutil, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import torch
import transformers
import datasets
import huggingface_hub
from scipy.ndimage import uniform_filter1d
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import (AutoConfig, AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)

assert torch.cuda.is_available(), "No GPU found: switch the Kaggle accelerator to a GPU."
assert torch.cuda.device_count() == 1, "More than one GPU is visible; the assignment allows exactly one."
DEVICE = torch.device("cuda")

VERSIONS = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "huggingface_hub": huggingface_hub.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_memory_GB": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
}
for k, v in VERSIONS.items():
    print(f"{k:>16}: {v}")

### Configuration

Every setting lives in this one cell.

- **`roberta-large`** is a strong encoder, and it suits this data well. Its byte-level tokenizer keeps emojis such as 😂 or ❤️ instead of turning them into an "unknown" token, and in Reddit comments the emoji often *is* the emotion.
- **Learning rate 2e-5 with a 6% warm-up, then linear decay.** Large models are touchy: too high a rate and training can collapse into predicting nothing. Warm-up eases the model in.
- **Layer-wise learning-rate decay (0.95 per layer).** Layers near the input hold general knowledge of English, so they move slowly. Layers near the output adapt faster, and the brand-new classification head learns 3× faster than the top layer.
- **3 epochs, scored 3 times per epoch, keeping the best snapshot.** Large models on this dataset usually peak within 2–3 epochs, often partway through one, and scoring validation takes only seconds.
- **`max_len = 128`.** Every batch is padded only to its own longest comment, so a generous limit costs nothing and no comment gets cut off.
- **Safety net.** If the model has collapsed after the first epoch (validation Macro-F1 below `COLLAPSE_F1`), training restarts once with the next seed.
- **`RUN_TRAINING`.** Set it to `False` to skip training and reuse the snapshot already saved in `run/`, for example to redo Parts 5–6 after a restart.

In [ ]:
SEED = 42
RUN_TRAINING = True

DATASET_NAME, DATASET_CONFIG = "google-research-datasets/go_emotions", "simplified"
MODEL_NAME = "FacebookAI/roberta-large"

CFG = dict(
    seed=SEED,
    max_len=128,
    batch_size=32,
    epochs=3,
    lr=2e-5,                 # peak learning rate of the top encoder layer
    llrd=0.95,               # layer-wise decay: each layer further down gets 0.95x the learning rate of the one above
    head_lr_mult=3.0,        # classification head learns 3x faster than the top encoder layer
    warmup_ratio=0.06,
    weight_decay=0.01,
    grad_clip=1.0,
    evals_per_epoch=3,
    patience=3,              # stop after this many validation checks without improvement
    fp16=True,
)
COLLAPSE_F1 = 0.25           # below this after epoch 1 = the run has collapsed; restart once with seed + 1

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./work")
RUN_DIR = WORK / "run"                   # training output: best snapshot, validation probabilities, history
FINAL_DIR = WORK / "goemotions_final"    # the submission folder (gets zipped)
WORK.mkdir(parents=True, exist_ok=True)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# Pin the exact commits of the base model and the dataset, so the run is reproducible.
_api = huggingface_hub.HfApi()
MODEL_REVISION = _api.model_info(MODEL_NAME).sha
DATASET_REVISION = _api.dataset_info(DATASET_NAME).sha
print(f"base model : {MODEL_NAME} @ {MODEL_REVISION}")
print(f"dataset    : {DATASET_NAME}/{DATASET_CONFIG} @ {DATASET_REVISION}")
print(json.dumps(CFG, indent=2))

## Part 1 · Data

We use the **simplified** GoEmotions configuration with its **official splits**, exactly as published. Each example is a comment plus a list of integer label IDs. We keep the original ID-to-name mapping (`neutral` is ID 27), and we keep comments where *neutral* appears together with another label exactly as annotated: no cleaning, no relabelling, no de-duplication.

We ask the library for **only the training and validation splits**. The test split never enters this session's memory until the optional last cell.

In [ ]:
train_ds, val_ds = load_dataset(DATASET_NAME, DATASET_CONFIG, revision=DATASET_REVISION,
                                split=["train", "validation"])

LABEL_NAMES = list(train_ds.features["labels"].feature.names)
NUM_LABELS = len(LABEL_NAMES)
NEUTRAL_ID = LABEL_NAMES.index("neutral")
assert NUM_LABELS == 28 and NEUTRAL_ID == 27
assert (len(train_ds), len(val_ds)) == (43410, 5426)

# datasets>=4 returns lazy Column objects; turn them into plain Python lists once, here.
train_texts, val_texts = list(train_ds["text"]), list(val_ds["text"])
train_label_ids, val_label_ids = list(train_ds["labels"]), list(val_ds["labels"])


def to_multihot(label_lists, n=NUM_LABELS):
    Y = np.zeros((len(label_lists), n), dtype=np.float32)
    for i, labs in enumerate(label_lists):
        Y[i, labs] = 1.0
    return Y


Y_train, Y_val = to_multihot(train_label_ids), to_multihot(val_label_ids)
print({"train": len(train_texts), "validation": len(val_texts)})
print(train_ds[0])

In [ ]:
stats = pd.DataFrame({
    "label": LABEL_NAMES,
    "train_n": Y_train.sum(0).astype(int),
    "val_n": Y_val.sum(0).astype(int),
})
stats["train_%"] = (100 * stats["train_n"] / len(Y_train)).round(2)

print(f"Average labels per comment (train):      {Y_train.sum(1).mean():.3f}")
print(f"Share of comments with 2 or more labels:  {(Y_train.sum(1) > 1).mean():.1%}")
print(f"'neutral' together with another label:    {int(((Y_train[:, NEUTRAL_ID] == 1) & (Y_train.sum(1) > 1)).sum())} train comments")
stats.sort_values("train_n", ascending=False).reset_index(drop=True)

**What the table says.** The labels are very unevenly spread. *Neutral* alone covers about a third of the comments, while *grief*, *pride*, *relief* and *nervousness* each cover well under 1%, and *grief* has only 13 examples in validation.

That matters because the score is **Macro-F1**, where every label counts equally: *grief* is worth exactly as much as *neutral*. A single shared cut-off such as 0.5 would almost never fire for the rare labels and would throw those points away. This is why the decision rule (Part 5) gives every label its own threshold.

### Tokenisation

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
PAD_ID = tokenizer.pad_token_id

lens = np.array([len(x) for x in tokenizer(train_texts)["input_ids"]])
print(f"tokens per comment: mean {lens.mean():.1f} | median {np.median(lens):.0f} | "
      f"95th pct {np.percentile(lens, 95):.0f} | 99th pct {np.percentile(lens, 99):.0f} | max {lens.max()}")
print(f"comments that fit in max_len={CFG['max_len']}: {(lens <= CFG['max_len']).mean():.2%}")

Comments are short (typically a dozen or two tokens), so `max_len = 128` keeps every comment whole.

**Batching, the main speed-up.** A batch has to be padded to the length of its longest comment. With fully random batches, one long comment forces the 31 short ones next to it to carry lots of padding, and the GPU spends much of its time on that padding. So we use **length-grouped batching** (the same idea as `group_by_length` in Hugging Face's `Trainer`):

1. shuffle the whole training set;
2. cut it into chunks of 50 batches' worth of comments, and sort each chunk by length;
3. slice each chunk into batches, then shuffle the *order* of the batches.

Every batch still holds a random mix of comments from across the dataset; they just have similar lengths. That removes most of the padding without changing what the model learns from. Padding itself is done by a tiny helper that training and inference share, so the two can never disagree about it.

In [ ]:
def encode(texts, max_len):
    return tokenizer(list(texts), truncation=True, max_length=max_len)["input_ids"]


def pad_batch(id_lists, pad_id):
    width = max(len(x) for x in id_lists)
    input_ids = torch.full((len(id_lists), width), pad_id, dtype=torch.long)
    attention_mask = torch.zeros((len(id_lists), width), dtype=torch.long)
    for i, ids in enumerate(id_lists):
        input_ids[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)
        attention_mask[i, :len(ids)] = 1
    return {"input_ids": input_ids, "attention_mask": attention_mask}


class CommentDataset(torch.utils.data.Dataset):
    def __init__(self, ids, Y):
        self.ids, self.Y = ids, Y

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return self.ids[i], self.Y[i]


class LengthGroupedBatchSampler(torch.utils.data.Sampler):
    # Shuffle, cut into chunks of `chunk_batches` batches, sort each chunk by length, slice into batches,
    # then shuffle the batch order. A new shuffle every epoch, reproducible from the seed.
    def __init__(self, lengths, batch_size, seed, chunk_batches=50):
        self.lengths = np.asarray(lengths)
        self.batch_size = batch_size
        self.chunk = batch_size * chunk_batches
        self.rng = np.random.default_rng(seed)

    def __iter__(self):
        perm = self.rng.permutation(len(self.lengths))
        batches = []
        for s in range(0, len(perm), self.chunk):
            chunk = perm[s:s + self.chunk]
            chunk = chunk[np.argsort(self.lengths[chunk], kind="stable")]
            batches += [chunk[i:i + self.batch_size].tolist() for i in range(0, len(chunk), self.batch_size)]
        for j in self.rng.permutation(len(batches)):
            yield batches[j]

    def __len__(self):
        return math.ceil(len(self.lengths) / self.batch_size)


def collate(batch):
    ids, ys = zip(*batch)
    out = pad_batch(ids, PAD_ID)
    out["labels"] = torch.from_numpy(np.stack(ys))
    return out


def make_train_loader(batch_size, seed):
    sampler = LengthGroupedBatchSampler([len(x) for x in enc_train], batch_size, seed)
    return torch.utils.data.DataLoader(CommentDataset(enc_train, Y_train), batch_sampler=sampler,
                                       collate_fn=collate, num_workers=2, pin_memory=True)


enc_train = encode(train_texts, CFG["max_len"])
enc_val = encode(val_texts, CFG["max_len"])

# How much padding does length grouping save? (count of token slots the GPU processes per epoch)
_loader = make_train_loader(CFG["batch_size"], CFG["seed"])
_grouped = sum(len(b) * max(len(enc_train[i]) for i in b) for b in _loader.batch_sampler)
_perm = np.random.default_rng(0).permutation(len(enc_train))
_random = sum(len(b) * max(len(enc_train[i]) for i in b)
              for b in np.array_split(_perm, math.ceil(len(_perm) / CFG["batch_size"])))
print(f"training batches per epoch: {len(_loader)}")
print(f"token slots per epoch: random batches {_random:,} vs length-grouped {_grouped:,} "
      f"({_random / _grouped:.1f}x less work), real tokens {sum(len(x) for x in enc_train):,}")
del _loader

## Part 2 · Model

We use Hugging Face's standard `AutoModelForSequenceClassification` with 28 outputs and `problem_type="multi_label_classification"`. For RoBERTa this means:

1. the encoder reads the comment;
2. the final hidden state of the first token (`<s>`, RoBERTa's "summary" token) goes through a small head: dense layer → tanh → dropout → linear layer;
3. the head produces **28 logits**, one per label, and each becomes a probability through its own **sigmoid**. Because there is no softmax, the labels don't compete and any number of them can be "on" at once. That is exactly what a multi-label task needs.

Using the library's standard class (rather than a hand-written head) means the saved checkpoint reloads in one line with no custom code, which makes the submission easy to verify. We load the weights explicitly in float32: mixed precision during training only *computes* in half precision, while the master copy of the weights stays in full precision.

### Why fine-tune the whole encoder instead of using LoRA or QLoRA?

Parameter-efficient methods are a natural question in a generative-AI course, so here is our reasoning:

- **What they are for.** LoRA trains small low-rank adapter matrices instead of the full weights, and QLoRA additionally stores the frozen weights in 4 bits. Both exist to make *billion-parameter* models fit in memory and cheaper to train. `roberta-large` has 355M parameters: full fine-tuning it with mixed precision uses well under half of the T4's 16 GB (the peak is logged in Part 4). The memory problem these methods solve does not exist here.
- **They save less time than you'd expect.** Each training step still runs the full forward and backward pass through all 24 layers, frozen or not. LoRA only skips the weight-gradient computations and the optimizer state for the frozen weights, so in our estimate it is at most about 1.5× faster per step. QLoRA has to unpack the 4-bit weights for every matrix multiplication, so on a T4 it is typically *slower* than plain mixed precision, and the 4-bit rounding adds noise.
- **Accuracy.** For encoders of this size on classification tasks, full fine-tuning generally matches or beats LoRA, and LoRA typically needs a higher learning rate and more epochs to catch up, which eats up its speed advantage.

The assignment leaves the fine-tuning method to us and grades Macro-F1, so we chose the method with the best expected score for the GPU time we have. Full fine-tuning also satisfies "the final solution must adapt the encoder" in the most direct way: every encoder weight is updated.

In [ ]:
def build_model():
    config = AutoConfig.from_pretrained(
        MODEL_NAME, revision=MODEL_REVISION,
        num_labels=NUM_LABELS,
        id2label={i: n for i, n in enumerate(LABEL_NAMES)},
        label2id={n: i for i, n in enumerate(LABEL_NAMES)},
        problem_type="multi_label_classification",
    )
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, revision=MODEL_REVISION, config=config, dtype=torch.float32)


def count_params(m):
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    head = sum(p.numel() for n, p in m.named_parameters() if n.startswith("classifier"))
    return {"total": total, "trainable": trainable, "classification_head": head, "encoder_and_embeddings": total - head}


# Built on the CPU just to count parameters; training builds its own fresh copy on the GPU.
_m = build_model()
PARAM_COUNTS = count_params(_m)
print(json.dumps(PARAM_COUNTS, indent=2))
print(f"trainable share: {PARAM_COUNTS['trainable'] / PARAM_COUNTS['total']:.0%} (full fine-tuning)")
print(_m.classifier)
del _m
gc.collect()

## Part 3 · Metrics and the decision-rule toolkit

These helpers are used both to monitor training and to build the final decision rule. They are plain NumPy, so they run in a blink.

- **`macro_f1`** follows the assignment's formula exactly: per label F1 = 2TP / (2TP + FP + FN), set to 0 when the denominator is 0, averaged over **all 28 labels**.
- **`f1_curves`**: for each label, F1 as a function of its threshold, on a grid from 0.01 to 0.90. The grid starts low on purpose: the best cut-off for a very rare label can sit well below 0.05.
- **`fit_thresholds`**: three ways to choose thresholds (explained in Part 5).
- **`apply_rule`**: turns probabilities into label sets and guarantees **at least one label per comment**.
- **`crossfit_table`**: an honest, validation-only estimate of how well a rule will do on unseen data (explained in Part 5).

A quick self-test on made-up data at the end checks the maths.

In [ ]:
GRID = np.round(np.arange(0.01, 0.901, 0.01), 2)


def f1_per_label(Y, P):
    Y, P = Y.astype(bool), P.astype(bool)
    tp, fp, fn = (Y & P).sum(0), (~Y & P).sum(0), (Y & ~P).sum(0)
    denom = 2 * tp + fp + fn
    return np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0)


def macro_f1(Y, P):
    return float(f1_per_label(Y, P).mean())


def micro_f1(Y, P):
    Y, P = Y.astype(bool), P.astype(bool)
    tp, fp, fn = (Y & P).sum(), (~Y & P).sum(), (Y & ~P).sum()
    return float(2 * tp / max(2 * tp + fp + fn, 1))


def f1_curves(Y, probs, grid=GRID):
    # (n_labels, n_thresholds) matrix: F1 of each label at each candidate threshold
    Yb = Y.astype(bool)
    curves = np.empty((Y.shape[1], len(grid)))
    for j in range(Y.shape[1]):
        pred = probs[:, j, None] >= grid[None, :]
        y = Yb[:, j, None]
        tp, fp, fn = (pred & y).sum(0), (pred & ~y).sum(0), (~pred & y).sum(0)
        d = 2 * tp + fp + fn
        curves[j] = np.where(d > 0, 2 * tp / np.maximum(d, 1), 0.0)
    return curves


def _middle_argmax(curve):
    # if several thresholds tie for the best score, take the middle one (steadier than the first)
    best = np.flatnonzero(curve >= curve.max() - 1e-12)
    return best[len(best) // 2]


def fit_thresholds(Y, probs, strategy="smooth_shrunk", smooth=5, k=30):
    curves = f1_curves(Y, probs)
    t_global = GRID[_middle_argmax(curves.mean(0))]
    if strategy == "global":
        return np.full(Y.shape[1], t_global)
    if strategy == "raw":
        return np.array([GRID[_middle_argmax(c)] for c in curves])
    if strategy != "smooth_shrunk":
        raise ValueError(strategy)
    # (1) smooth each label's F1-vs-threshold curve so one lucky spike can't win;
    # (2) pull the threshold toward the global one, more strongly when the label has few positives.
    smoothed = uniform_filter1d(curves, size=smooth, axis=1, mode="nearest")
    t_label = np.array([GRID[_middle_argmax(c)] for c in smoothed])
    n_pos = Y.sum(0)
    w = n_pos / (n_pos + k)
    return w * t_label + (1 - w) * t_global


def apply_rule(probs, thr, fallback="ratio", neutral_id=NEUTRAL_ID):
    # probs (N, L), thr (L,) -> boolean (N, L) with at least one True per row
    pred = probs >= thr[None, :]
    empty = np.flatnonzero(~pred.any(1))
    if len(empty):
        if fallback == "ratio":      # the label that comes closest to its own threshold
            idx = (probs[empty] / thr[None, :]).argmax(1)
        elif fallback == "prob":     # simply the most probable label
            idx = probs[empty].argmax(1)
        elif fallback == "neutral":
            idx = np.full(len(empty), neutral_id)
        else:
            raise ValueError(fallback)
        pred[empty, idx] = True
    return pred


STRATEGIES = {
    "global":                {"strategy": "global"},
    "raw":                   {"strategy": "raw"},
    "smooth_shrunk (k=10)":  {"strategy": "smooth_shrunk", "smooth": 5, "k": 10},
    "smooth_shrunk (k=30)":  {"strategy": "smooth_shrunk", "smooth": 5, "k": 30},
    "smooth_shrunk (k=100)": {"strategy": "smooth_shrunk", "smooth": 5, "k": 100},
}


def crossfit_score(Y, probs, repeats=40, seed=0, **fit_kw):
    # Split validation in half at random, fit thresholds on one half, score on the other; repeat both ways.
    rng = np.random.default_rng(seed)
    scores = []
    for _ in range(repeats):
        perm = rng.permutation(len(Y))
        a, b = perm[: len(Y) // 2], perm[len(Y) // 2:]
        for fit_idx, score_idx in ((a, b), (b, a)):
            thr = fit_thresholds(Y[fit_idx], probs[fit_idx], **fit_kw)
            scores.append(macro_f1(Y[score_idx], apply_rule(probs[score_idx], thr)))
    return float(np.mean(scores)), float(np.std(scores))


def crossfit_table(Y, probs):
    rows = []
    for name, kw in STRATEGIES.items():
        mean, std = crossfit_score(Y, probs, **kw)
        rows.append({"strategy": name, "heldout_macro_f1": round(mean, 4), "std": round(std, 4)})
    return pd.DataFrame(rows).sort_values("heldout_macro_f1", ascending=False).reset_index(drop=True)


# Self-test on made-up data
_rng = np.random.default_rng(0)
_Y = (_rng.random((600, 4)) < [0.5, 0.2, 0.05, 0.01]).astype(np.float32)
_P = np.clip(_Y * 0.6 + _rng.random((600, 4)) * 0.4, 0, 1)
assert apply_rule(_P, np.full(4, 0.99)).any(1).all()                  # fallback guarantees >= 1 label
assert abs(macro_f1(_Y, _Y) - np.mean(_Y.sum(0) > 0)) < 1e-9           # perfect predictions; empty labels score 0
for _kw in STRATEGIES.values():
    _t = fit_thresholds(_Y, _P, **_kw)
    assert _t.shape == (4,) and (_t >= GRID[0]).all() and (_t <= GRID[-1]).all()
print("toolkit self-test passed")

## Part 4 · Training

The choices a reader is most likely to ask about:

- **Loss: binary cross-entropy on the logits** (`BCEWithLogitsLoss`), one yes/no problem per label. We don't re-weight labels: the per-label thresholds in Part 5 already handle the imbalance, and re-weighting tends to make the probabilities less well-behaved.
- **Mixed precision (fp16).** Roughly halves memory and speeds up the T4 a lot. The loss itself is computed in float32 for numerical safety, and a gradient scaler stops tiny gradients from rounding to zero.
- **AdamW with warm-up, linear decay, and gradient clipping at 1.0.** The standard, stable recipe for fine-tuning Transformers.
- **Choosing the best snapshot.** Three times per epoch we score the validation set using Macro-F1 under the *best single shared threshold*. That score has only one tuned number, so it is hard to overfit, which makes it a fair way to compare snapshots. The full per-label rule is tuned afterwards, once the snapshot is fixed.
- **Safety net.** The collapse restart described in Part 0. A collapsed model predicts almost nothing, and its Macro-F1 sits far below `COLLAPSE_F1`, so the check cannot be triggered by a merely mediocre run.

Training leaves behind, in `run/`, the best snapshot's weights and tokenizer, its validation probabilities (so Part 5 needs no GPU), the training history, and `train_info.json`.

In [ ]:
def build_optimizer(model, cfg):
    no_decay = ("bias", "LayerNorm", "layer_norm")
    n_layers = model.config.num_hidden_layers
    named = list(model.named_parameters())
    groups, used = [], set()

    def add(items, lr):
        for n, _ in items:
            assert n not in used, f"{n} assigned twice"
            used.add(n)
        decay = [p for n, p in items if not any(nd in n for nd in no_decay)]
        nodecay = [p for n, p in items if any(nd in n for nd in no_decay)]
        if decay:
            groups.append({"params": decay, "lr": lr, "weight_decay": cfg["weight_decay"]})
        if nodecay:
            groups.append({"params": nodecay, "lr": lr, "weight_decay": 0.0})

    lr, llrd = cfg["lr"], cfg["llrd"]
    add([(n, p) for n, p in named if n.startswith("classifier")], lr * cfg["head_lr_mult"])
    for i in range(n_layers):                                    # top layer (n_layers-1) gets lr, lower layers less
        add([(n, p) for n, p in named if f"encoder.layer.{i}." in n], lr * llrd ** (n_layers - 1 - i))
    add([(n, p) for n, p in named if "embeddings" in n], lr * llrd ** n_layers)
    leftovers = [(n, p) for n, p in named if n not in used]
    if leftovers:
        add(leftovers, lr)
    assert len(used) == len(named)
    return torch.optim.AdamW(groups, betas=(0.9, 0.999), eps=1e-8, fused=True)


@torch.inference_mode()
def predict_probs(model, id_lists, fp16=True, batch_size=128):
    model.eval()
    order = np.argsort([len(x) for x in id_lists])           # similar lengths together = minimal padding
    out = np.zeros((len(id_lists), NUM_LABELS), dtype=np.float32)
    for s in range(0, len(order), batch_size):
        idx = order[s:s + batch_size]
        batch = {k: v.to(DEVICE) for k, v in pad_batch([id_lists[i] for i in idx], PAD_ID).items()}
        with torch.autocast("cuda", dtype=torch.float16, enabled=fp16):
            logits = model(**batch).logits
        out[idx] = torch.sigmoid(logits.float()).cpu().numpy()
    return out


def quick_val_scores(Y, probs):
    curves = f1_curves(Y, probs)
    g = _middle_argmax(curves.mean(0))
    return {
        "macro_f1_at_0.5": macro_f1(Y, probs >= 0.5),
        "macro_f1_best_global": float(curves.mean(0)[g]),
        "best_global_threshold": float(GRID[g]),
    }

In [ ]:
def train(cfg):
    # One training attempt. Returns its info dict; info["collapsed"] is True if it was aborted as collapsed.
    if RUN_DIR.exists():
        shutil.rmtree(RUN_DIR)
    (RUN_DIR / "model").mkdir(parents=True)

    set_seed(cfg["seed"])
    torch.cuda.reset_peak_memory_stats()
    t_start = time.time()

    model = build_model().to(DEVICE)
    loader = make_train_loader(cfg["batch_size"], cfg["seed"])
    optimizer = build_optimizer(model, cfg)
    total_steps = len(loader) * cfg["epochs"]
    scheduler = get_linear_schedule_with_warmup(optimizer, int(cfg["warmup_ratio"] * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg["fp16"])
    loss_fn = torch.nn.BCEWithLogitsLoss()
    eval_points = {round(len(loader) * k / cfg["evals_per_epoch"]) for k in range(1, cfg["evals_per_epoch"] + 1)}

    history, best, checks_without_gain, step = [], {"score": -1.0, "step": None, "epoch": None}, 0, 0
    stop_reason, collapsed = "finished all epochs", False

    for epoch in range(1, cfg["epochs"] + 1):
        model.train()
        recent = []
        pbar = tqdm(loader, desc=f"seed {cfg['seed']} | epoch {epoch}/{cfg['epochs']}")
        for i, batch in enumerate(pbar, start=1):
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            labels = batch.pop("labels")
            with torch.autocast("cuda", dtype=torch.float16, enabled=cfg["fp16"]):
                logits = model(**batch).logits
            loss = loss_fn(logits.float(), labels)              # loss in float32
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            step += 1
            recent.append(loss.item())
            if i % 50 == 0:
                pbar.set_postfix(loss=f"{np.mean(recent[-50:]):.4f}")

            if i in eval_points:
                val_probs = predict_probs(model, enc_val, fp16=cfg["fp16"])
                scores = quick_val_scores(Y_val, val_probs)
                rec = {"step": step, "epoch": round(step / len(loader), 2),
                       "train_loss": float(np.mean(recent[-200:])), **scores}
                history.append(rec)
                if scores["macro_f1_best_global"] > best["score"] + 1e-5:
                    best = {"score": scores["macro_f1_best_global"], "step": step, "epoch": rec["epoch"]}
                    checks_without_gain = 0
                    model.save_pretrained(RUN_DIR / "model")
                    tokenizer.save_pretrained(RUN_DIR / "model")
                    np.save(RUN_DIR / "val_probs.npy", val_probs)
                    tag = "new best, saved"
                else:
                    checks_without_gain += 1
                    tag = f"no gain ({checks_without_gain}/{cfg['patience']})"
                elapsed = (time.time() - t_start) / 60
                tqdm.write(f"  epoch {rec['epoch']:.2f} | {elapsed:5.1f} min | train loss {rec['train_loss']:.4f} | "
                           f"val macro-F1 {scores['macro_f1_best_global']:.4f} @ t={scores['best_global_threshold']:.2f} "
                           f"| @0.5 {scores['macro_f1_at_0.5']:.4f} | {tag}")
                model.train()
                if checks_without_gain >= cfg["patience"]:
                    stop_reason = "early stopping (no improvement)"
                elif epoch == 1 and i == len(loader) and best["score"] < COLLAPSE_F1:
                    stop_reason, collapsed = "collapsed after epoch 1", True
                if stop_reason != "finished all epochs":
                    break
        if stop_reason != "finished all epochs":
            break

    info = {
        "seed": cfg["seed"], "collapsed": collapsed, "stop_reason": stop_reason,
        "best_step": best["step"], "best_epoch": best["epoch"], "val_macro_f1_best_global": best["score"],
        "minutes": round((time.time() - t_start) / 60, 1),
        "peak_gpu_memory_GB": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    }
    json.dump(history, open(RUN_DIR / "history.json", "w"), indent=2)
    json.dump(info, open(RUN_DIR / "train_info.json", "w"), indent=2)
    print(f"  stopped: {stop_reason} | best val macro-F1 {best['score']:.4f} at epoch {best['epoch']} | "
          f"{info['minutes']} min | peak GPU memory {info['peak_gpu_memory_GB']} GB")

    del model, optimizer, scheduler, scaler
    gc.collect()
    torch.cuda.empty_cache()
    return info

In [ ]:
if RUN_TRAINING:
    TRAIN_INFO = train(CFG)
    if TRAIN_INFO["collapsed"]:
        print(f"\nThe run collapsed; restarting once with seed {CFG['seed'] + 1}.")
        CFG = {**CFG, "seed": CFG["seed"] + 1}
        TRAIN_INFO = train(CFG)
        assert not TRAIN_INFO["collapsed"], "Training collapsed twice; lower CFG['lr'] and run again."
else:
    TRAIN_INFO = json.load(open(RUN_DIR / "train_info.json"))
    CFG = {**CFG, "seed": TRAIN_INFO["seed"]}
    print("RUN_TRAINING=False: reusing the saved snapshot in", RUN_DIR)
print(json.dumps(TRAIN_INFO, indent=2))

In [ ]:
h = pd.DataFrame(json.load(open(RUN_DIR / "history.json")))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(h.epoch, h.train_loss, marker="o")
ax[0].set(title="Training loss", xlabel="epoch")
ax[1].plot(h.epoch, h.macro_f1_best_global, marker="o", label="best shared threshold")
ax[1].plot(h.epoch, h["macro_f1_at_0.5"], marker="o", label="threshold 0.5")
ax[1].set(title="Validation macro-F1", xlabel="epoch")
ax[1].legend()
plt.tight_layout()
plt.show()
display(h)

**How to read the curves.** The training loss should fall steadily. Validation Macro-F1 usually climbs fast in the first epoch and flattens in the second or third; the saved snapshot is the highest point of the blue line.

Note the gap between the two lines: with the default cut-off of 0.5, the rare labels hardly ever fire. That gap is why the decision rule in Part 5 matters so much.

## Part 5 · The decision rule (tuned on validation only)

The model gives 28 probabilities per comment. We still have to decide **which** labels to return and therefore **how many**. Our rule: a label is returned if its probability reaches its own threshold. A comment can therefore get one label, several, or none, and "none" is handled by a fallback.

Macro-F1 is an average of separate per-label scores, so choosing one threshold per label is exactly the right lever. The only danger is noise: a label with 13 validation examples can't tell us its best threshold very precisely.

**Step A: how to choose the thresholds.** Three options:

1. **global**: one threshold shared by all labels.
2. **raw**: each label gets the threshold that maximises its own validation F1.
3. **smooth_shrunk**: like *raw*, but (a) each label's F1-versus-threshold curve is smoothed first, so a single lucky spike can't decide, and (b) the threshold is pulled toward the global one in proportion to how few examples the label has (weight `n / (n + k)`). A classic way to stop rare labels from chasing noise.

**Why we can't just compare them on validation directly.** Thresholds tuned on a set always look great on that same set, and *raw* would win every time. So we **cross-fit**: split validation in half at random, fit thresholds on one half, score them on the other half, swap the halves, and repeat 40 times. The strategy with the best *held-out* average wins. Validation only; the test set is never touched.

**Step B: the fallback** for comments where no label reaches its threshold. Every comment must get at least one label. The candidates are: the label whose probability is highest *relative to its own threshold* (`ratio`), simply the most probable label (`prob`), or *neutral*.

In [ ]:
val_probs = np.load(RUN_DIR / "val_probs.npy")
assert val_probs.shape == Y_val.shape

cmp_df = crossfit_table(Y_val, val_probs)
display(cmp_df)

BEST_STRATEGY = cmp_df.loc[0, "strategy"]
BEST_KW = STRATEGIES[BEST_STRATEGY]
print("chosen threshold strategy:", BEST_STRATEGY)

In [ ]:
THR = fit_thresholds(Y_val, val_probs, **BEST_KW)        # final thresholds: fitted on the whole validation set

empty_share = (~(val_probs >= THR[None, :]).any(1)).mean()
print(f"validation comments where no label reaches its threshold: {empty_share:.1%}")

fb_rows = []
for fb in ("ratio", "prob", "neutral"):
    P = apply_rule(val_probs, THR, fb)
    fb_rows.append({"fallback": fb, "val macro-F1": round(macro_f1(Y_val, P), 4), "val micro-F1": round(micro_f1(Y_val, P), 4)})
display(pd.DataFrame(fb_rows))

FALLBACK = max(fb_rows, key=lambda r: r["val macro-F1"])["fallback"]
print("chosen fallback:", FALLBACK)

In [ ]:
P_val = apply_rule(val_probs, THR, FALLBACK)
per_label = pd.DataFrame({
    "label": LABEL_NAMES,
    "val_support": Y_val.sum(0).astype(int),
    "threshold": THR.round(3),
    "F1": f1_per_label(Y_val, P_val).round(3),
}).sort_values("F1", ascending=False).reset_index(drop=True)

VAL_METRICS = {
    "macro_f1_at_0.5": macro_f1(Y_val, apply_rule(val_probs, np.full(NUM_LABELS, 0.5), FALLBACK)),
    "macro_f1_tuned_in_sample": macro_f1(Y_val, P_val),
    "micro_f1_tuned_in_sample": micro_f1(Y_val, P_val),
    "macro_f1_crossfit_heldout": float(cmp_df.loc[0, "heldout_macro_f1"]),
    "avg_labels_predicted": float(P_val.sum(1).mean()),
    "avg_labels_true": float(Y_val.sum(1).mean()),
}
print(json.dumps({k: round(v, 4) for k, v in VAL_METRICS.items()}, indent=2))
display(per_label)

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.bar(per_label.label, per_label.F1)
ax.set_xticks(range(len(per_label)))
ax.set_xticklabels(per_label.label, rotation=75, ha="right")
ax.set(ylabel="validation F1", title="Per-label F1 on validation")
plt.tight_layout()
plt.show()

**How to read the result.** Labels with distinctive wording (*gratitude*: "thanks"; *amusement*: "lol"; *love*) are easy. Labels that are rare, subtle, or both (*relief*, *grief*, *realization*, *disappointment*) are hard. That spread is normal for GoEmotions, and it's why Macro-F1 sits well below micro-F1.

The **honest** estimate of performance on unseen data is the cross-fitted held-out Macro-F1. The in-sample number is computed on the same comments the thresholds were tuned on, so it is an optimistic upper bound.

## Part 6 · Saving the final checkpoint

We copy the best snapshot's weights and tokenizer into a clean folder, `goemotions_final/`, add everything inference needs, and zip it:

| File | Contents |
|---|---|
| `model.safetensors`, `config.json` | the complete fine-tuned model (full fine-tuning, so no adapters and no separate base-model download are needed); `config.json` also holds the label mapping |
| tokenizer files | the exact tokenizer used in training |
| `prediction_rule.json` | the 28 thresholds, the fallback rule, the label order, and the max sequence length |
| `label_mapping.json` | the original GoEmotions ID → name mapping |
| `run_config.json` | base model ID **and revision**, dataset revision, seed, final configuration, parameter counts, library versions, training summary, validation metrics |
| `FINAL_CHECKPOINT.txt` | marks this folder as the one to evaluate; the inference section looks for this file |

In [ ]:
if FINAL_DIR.exists():
    shutil.rmtree(FINAL_DIR)
shutil.copytree(RUN_DIR / "model", FINAL_DIR)

json.dump({
    "label_order": LABEL_NAMES,
    "thresholds": {n: float(t) for n, t in zip(LABEL_NAMES, THR)},
    "fallback": FALLBACK,
    "threshold_strategy": BEST_STRATEGY, "threshold_strategy_params": BEST_KW,
    "max_len": CFG["max_len"],
    "fp16_inference": CFG["fp16"],
}, open(FINAL_DIR / "prediction_rule.json", "w"), indent=2)

json.dump({str(i): n for i, n in enumerate(LABEL_NAMES)}, open(FINAL_DIR / "label_mapping.json", "w"), indent=2)

json.dump({
    "base_model": MODEL_NAME, "base_model_revision": MODEL_REVISION,
    "dataset_name": DATASET_NAME, "dataset_config": DATASET_CONFIG, "dataset_revision": DATASET_REVISION,
    "seed": CFG["seed"],
    "final_config": CFG,
    "training": TRAIN_INFO,
    "param_counts": PARAM_COUNTS,
    "versions": VERSIONS,
    "validation_metrics": VAL_METRICS,
}, open(FINAL_DIR / "run_config.json", "w"), indent=2)

(FINAL_DIR / "FINAL_CHECKPOINT.txt").write_text("\n".join([
    "FINAL CHECKPOINT TO BE EVALUATED: this folder (goemotions_final).",
    f"Base model: {MODEL_NAME} @ revision {MODEL_REVISION} (full fine-tuning: model.safetensors is the complete model).",
    "Load: AutoModelForSequenceClassification.from_pretrained(<this folder>) and AutoTokenizer.from_pretrained(<this folder>).",
    "Decision rule: per-label sigmoid thresholds + fallback, stored in prediction_rule.json.",
    "The notebook's inference section (Part 7) loads this folder and exposes predict(texts).",
]) + "\n")

zip_path = shutil.make_archive(str(WORK / "goemotions_final"), "zip", root_dir=str(WORK), base_dir=FINAL_DIR.name)

sha = hashlib.sha256()
with open(FINAL_DIR / "model.safetensors", "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        sha.update(chunk)

print("=" * 72)
print("FINAL SUBMISSION")
print(f"  base model         : {MODEL_NAME} @ {MODEL_REVISION}")
print(f"  seed               : {CFG['seed']}")
print(f"  parameters         : {PARAM_COUNTS['total']:,} total, {PARAM_COUNTS['trainable']:,} trainable")
print(f"  best snapshot      : epoch {TRAIN_INFO['best_epoch']} ({TRAIN_INFO['stop_reason']})")
print(f"  held-out macro-F1  : {VAL_METRICS['macro_f1_crossfit_heldout']:.4f} (validation, cross-fitted)")
print(f"  weights sha256     : {sha.hexdigest()}")
print(f"  archive            : {zip_path} ({os.path.getsize(zip_path) / 1e9:.2f} GB)")
print(f"  files              : {sorted(p.name for p in FINAL_DIR.iterdir())}")
print("=" * 72)

---
## Part 7 · Inference section (self-contained)

**Nothing in this section depends on the cells above.** To use it:

1. Start a fresh session (or restart the kernel).
2. Make the checkpoint available: either `goemotions_final.zip` or the unzipped `goemotions_final/` folder, for example attached as a Kaggle dataset under `/kaggle/input/...` or placed in the working directory. The code finds it automatically by looking for `FINAL_CHECKPOINT.txt`. If it doesn't, set `CKPT_PATH` by hand.
3. Run only the cells below. A GPU is used if one is present; otherwise everything runs (more slowly) on the CPU.

The section exposes **`predict(texts)`**: it takes a list of strings and returns, for each input and in the same order, a list of unique label names. Every input gets at least one label.

The next cell wipes every variable created so far. When this notebook runs top to bottom, that proves the inference code does not secretly rely on anything from training. In a fresh session it does nothing.

In [ ]:
%reset -f

In [ ]:
# ===================== INFERENCE: everything needed is imported and defined here =====================
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")       # one GPU at most

import gc, glob, json, tempfile, zipfile
from pathlib import Path

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

CKPT_PATH = None                  # optional: path to goemotions_final.zip or to the goemotions_final folder
MARKER = "FINAL_CHECKPOINT.txt"


def _locate_checkpoint():
    if CKPT_PATH:
        return Path(CKPT_PATH)
    for folder in (Path("/kaggle/working/goemotions_final"), Path("goemotions_final")):
        if (folder / MARKER).exists():
            return folder
    for root in ("/kaggle/input", "."):
        hits = sorted(glob.glob(os.path.join(root, "**", MARKER), recursive=True))
        if hits:
            return Path(hits[0]).parent
        zips = sorted(glob.glob(os.path.join(root, "**", "goemotions_final*.zip"), recursive=True))
        if zips:
            return Path(zips[0])
    raise FileNotFoundError("Checkpoint not found. Set CKPT_PATH to goemotions_final.zip or its unzipped folder.")


def _as_checkpoint_folder(path):
    path = Path(path)
    if path.suffix == ".zip":
        dest = Path(tempfile.mkdtemp(prefix="goemotions_"))
        with zipfile.ZipFile(path) as zf:
            zf.extractall(dest)
        path = dest
    if (path / MARKER).exists():
        return path
    hits = sorted(glob.glob(str(path / "**" / MARKER), recursive=True))
    if not hits:
        raise FileNotFoundError(f"No {MARKER} inside {path}")
    return Path(hits[0]).parent


gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

CKPT_DIR = _as_checkpoint_folder(_locate_checkpoint())
print("loading checkpoint from:", CKPT_DIR)

_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_tokenizer = AutoTokenizer.from_pretrained(CKPT_DIR)
_model = AutoModelForSequenceClassification.from_pretrained(CKPT_DIR, dtype=torch.float32).to(_device).eval()
_rule = json.load(open(CKPT_DIR / "prediction_rule.json"))

_label_names = [_model.config.id2label[i] for i in range(_model.config.num_labels)]
assert _label_names == _rule["label_order"], "label order in the model config and the decision rule disagree"
_thresholds = np.array([_rule["thresholds"][n] for n in _label_names], dtype=np.float32)
_max_len, _fallback = _rule["max_len"], _rule["fallback"]
_use_fp16 = _rule["fp16_inference"] and _device.type == "cuda"     # same numeric setting the thresholds were tuned with
_neutral_id = _label_names.index("neutral")


def _pad(id_lists, pad_id):
    width = max(len(x) for x in id_lists)
    input_ids = torch.full((len(id_lists), width), pad_id, dtype=torch.long)
    attention_mask = torch.zeros((len(id_lists), width), dtype=torch.long)
    for i, ids in enumerate(id_lists):
        input_ids[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)
        attention_mask[i, :len(ids)] = 1
    return {"input_ids": input_ids, "attention_mask": attention_mask}


@torch.inference_mode()
def _predict_probs(texts, batch_size=128):
    ids = _tokenizer(texts, truncation=True, max_length=_max_len)["input_ids"]
    order = np.argsort([len(x) for x in ids])
    out = np.zeros((len(ids), len(_label_names)), dtype=np.float32)
    for s in range(0, len(order), batch_size):
        idx = order[s:s + batch_size]
        batch = {k: v.to(_device) for k, v in _pad([ids[i] for i in idx], _tokenizer.pad_token_id).items()}
        with torch.autocast(_device.type, dtype=torch.float16, enabled=_use_fp16):
            logits = _model(**batch).logits
        out[idx] = torch.sigmoid(logits.float()).cpu().numpy()
    return out


def _decide(probs):
    pred = probs >= _thresholds[None, :]
    empty = np.flatnonzero(~pred.any(1))
    if len(empty):
        if _fallback == "ratio":
            idx = (probs[empty] / _thresholds[None, :]).argmax(1)
        elif _fallback == "prob":
            idx = probs[empty].argmax(1)
        elif _fallback == "neutral":
            idx = np.full(len(empty), _neutral_id)
        else:
            raise ValueError(_fallback)
        pred[empty, idx] = True
    return pred


def predict(texts):
    """list[str] -> list[list[str]]: unique label names per input, same order as the input, at least one each."""
    if isinstance(texts, str):
        raise TypeError("predict() expects a list of strings, e.g. predict(['I love this!'])")
    texts = [str(t) for t in texts]
    if not texts:
        return []
    probs = _predict_probs(texts)
    chosen = _decide(probs)
    results = []
    for p, row in zip(probs, chosen):
        idx = np.flatnonzero(row)
        idx = idx[np.argsort(-p[idx])]          # most confident first (order is not scored)
        results.append([_label_names[i] for i in idx])
    return results


demo = ["Thank you so much, this really made my day!",
        "I can't believe he did that, I'm furious.",
        "Not sure what to think about this.",
        "I miss her so much it hurts.",
        "lol that's hilarious 😂"]
for text, labels in zip(demo, predict(demo)):
    print(f"{labels}  <-  {text}")

### Sanity check: does the reloaded checkpoint reproduce the validation score?

This reloads the **validation** split (not the test split), runs it through the public `predict()` function, and compares the Macro-F1 with the number recorded at training time. They should agree to within a small rounding difference from half-precision arithmetic on the GPU. It also checks the output contract: one answer per input, at least one label each, and no duplicates.

In [ ]:
from datasets import load_dataset

_run = json.load(open(CKPT_DIR / "run_config.json"))
_val = load_dataset(_run["dataset_name"], _run["dataset_config"], revision=_run["dataset_revision"], split="validation")
_texts = list(_val["text"])

_out = predict(_texts)
assert len(_out) == len(_texts)
assert all(len(r) >= 1 and len(r) == len(set(r)) for r in _out)

_Y = np.zeros((len(_val), len(_label_names)), dtype=bool)
for i, labs in enumerate(_val["labels"]):
    _Y[i, labs] = True
_P = np.array([[n in set(r) for n in _label_names] for r in _out])
_tp, _fp, _fn = (_Y & _P).sum(0), (~_Y & _P).sum(0), (_Y & ~_P).sum(0)
_den = 2 * _tp + _fp + _fn
reloaded = float(np.where(_den > 0, 2 * _tp / np.maximum(_den, 1), 0.0).mean())
recorded = _run["validation_metrics"]["macro_f1_tuned_in_sample"]
print(f"reloaded checkpoint via predict(), validation macro-F1: {reloaded:.4f}")
print(f"recorded at training time (in-sample):                  {recorded:.4f}")
assert abs(reloaded - recorded) < 0.01, "reloaded model does not reproduce the recorded validation score"

---
## Part 8 · One-time test evaluation (off by default)

It reuses the checkpoint and helpers loaded in Part 7, so run Part 7 first. Everything is frozen at this point: the checkpoint, the thresholds and the fallback. Flip the flag to `True` and run this cell **once**, at the very end, purely to report a number. Nothing is changed afterwards based on it.

In [ ]:
RUN_TEST_EVAL = False

if RUN_TEST_EVAL:
    _test = load_dataset(_run["dataset_name"], _run["dataset_config"], revision=_run["dataset_revision"], split="test")
    _Yt = np.zeros((len(_test), len(_label_names)), dtype=bool)
    for i, labs in enumerate(_test["labels"]):
        _Yt[i, labs] = True
    _Pt = np.array([[n in set(r) for n in _label_names] for r in predict(list(_test["text"]))])
    tp, fp, fn = (_Yt & _Pt).sum(0), (~_Yt & _Pt).sum(0), (_Yt & ~_Pt).sum(0)
    f1 = np.where((2 * tp + fp + fn) > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0)
    print(f"TEST macro-F1: {f1.mean():.4f}")
    print(f"TEST micro-F1: {2 * tp.sum() / (2 * tp.sum() + fp.sum() + fn.sum()):.4f}")
else:
    print("Test evaluation skipped (RUN_TEST_EVAL = False).")